# The Power of Consensus: Why Multiple Judges Matter

**Goal:** Understand why a jury of models is more reliable than a single judge.

In this tutorial, you'll discover:
1. How different models can have varying perspectives
2. Why consensus reduces bias and improves reliability
3. How to interpret inter-judge agreement
4. When disagreement is actually valuable information

---

## The Challenge

A single LLM judge can be:
- **Biased** toward certain writing styles
- **Inconsistent** across similar inputs
- **Over-confident** in incorrect assessments
- **Model-specific** in its interpretation

By using **multiple judges**, we get:
- ✅ Reduced individual model bias
- ✅ More robust evaluations
- ✅ Confidence metrics from agreement levels
- ✅ Transparency into reasoning diversity

---

## 📦 Setup

We'll use judges from **multiple providers** to show true diversity:
- **Groq**: Fast inference with Llama models
- **OpenAI**: Industry-standard GPT models
- **Google**: Gemini models

```bash
pip install llm-jury
pip install langchain-groq langchain-openai langchain-google-genai
```

In [ ]:
# API Keys Configuration
import os

# Set your API keys here (or use environment variables)
GROQ_API_KEY = "gsk_YOUR_GROQ_KEY"
OPENAI_API_KEY = "sk-YOUR_OPENAI_KEY"
GOOGLE_API_KEY = "YOUR_GOOGLE_KEY"

# Or load from environment
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [3]:
# Import all required components
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from llm_jury.core.evaluator import JuryEvaluator
from llm_jury.judges.llm_judge import LLMJudge
from llm_jury.metrics.predefined import GroundednessMetric
from llm_jury.strategies.consensus import MajorityVoting

import pandas as pd
from IPython.display import display, HTML

print("✅ All imports successful!")

✅ All imports successful!


## 🏗️ Building a Diverse Jury

Let's create **5 judges** from different providers and model families:

1. **Llama 3.3 70B** (Groq) - Large, capable open model
2. **Llama 3.1 8B** (Groq) - Smaller, faster model
3. **GPT-4o** (OpenAI) - Advanced reasoning
4. **GPT-3.5 Turbo** (OpenAI) - Efficient baseline
5. **Gemini 1.5 Flash** (Google) - Fast Google model

This diversity ensures we're not just getting echo chambers!

In [4]:
print("🏗️ Initializing Judge Panel...\n")

# Judge 1: Llama 3.3 70B (Groq)
judge_1 = LLMJudge(
    model=ChatGroq(
        model_name="llama-3.3-70b-versatile",
        temperature=0.0,
        api_key=GROQ_API_KEY
    ),
    name="Llama-3.3-70B"
)
print(f"✅ Judge 1: {judge_1.name} (Groq)")

# Judge 2: Llama 3.1 8B (Groq)
judge_2 = LLMJudge(
    model=ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.0,
        api_key=GROQ_API_KEY
    ),
    name="Llama-3.1-8B"
)
print(f"✅ Judge 2: {judge_2.name} (Groq)")

# Judge 3: GPT-4o (OpenAI)
judge_3 = LLMJudge(
    model=ChatOpenAI(
        model="gpt-4o",
        temperature=0.0,
        api_key=OPENAI_API_KEY
    ),
    name="GPT-4o"
)
print(f"✅ Judge 3: {judge_3.name} (OpenAI)")

# Judge 4: GPT-3.5 Turbo (OpenAI)
judge_4 = LLMJudge(
    model=ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0.0,
        api_key=OPENAI_API_KEY
    ),
    name="GPT-3.5-Turbo"
)
print(f"✅ Judge 4: {judge_4.name} (OpenAI)")

# Judge 5: Gemini 2.5 Flash (Google)
judge_5 = LLMJudge(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.0,
        google_api_key=GOOGLE_API_KEY
    ),
    name="Gemini-2.5-Flash"
)
print(f"✅ Judge 5: {judge_5.name} (Google)")

print(f"\n🎉 Successfully initialized {5} judges from 3 different providers!")

🏗️ Initializing Judge Panel...

✅ Judge 1: Llama-3.3-70B (Groq)
✅ Judge 2: Llama-3.1-8B (Groq)
✅ Judge 3: GPT-4o (OpenAI)
✅ Judge 4: GPT-3.5-Turbo (OpenAI)
✅ Judge 5: Gemini-2.5-Flash (Google)

🎉 Successfully initialized 5 judges from 3 different providers!


## ⚖️ Create the Jury with Majority Voting

We'll use **MajorityVoting** strategy, which:
- Selects the most frequent score
- Calculates confidence based on agreement ratio
- Handles ties gracefully

In [5]:
# Create the jury
jury = JuryEvaluator(
    judges=[judge_1, judge_2, judge_3, judge_4, judge_5],
    strategy=MajorityVoting()
)

print(f"⚖️ Jury assembled with {len(jury.judges)} judges")
print(f"📊 Strategy: {jury.strategy.__class__.__name__}")

⚖️ Jury assembled with 5 judges
📊 Strategy: MajorityVoting


## 🎭 The Ambiguous Test Case

We'll craft an **intentionally ambiguous** example where:
- The output is *partially* grounded
- Some facts are correct, others are inferred
- Reasonable judges might disagree

This will show us the **diversity of judge reasoning** and the value of consensus.

In [6]:
# Source context: Technical documentation about a feature
source_text = """
The new API endpoint /v2/users/search accepts query parameters 'name' and 'email'. 
It returns a JSON array of matching user objects. The endpoint requires authentication 
via Bearer token. Rate limiting is set to 100 requests per hour per API key.
"""

# Model output: Mixes explicit facts with reasonable inferences
output_text = """
The /v2/users/search endpoint allows you to find users by name or email address. 
You must include a valid authentication token in the Authorization header. 
The endpoint is optimized for performance and returns results in JSON format. 
Keep in mind the rate limit of 100 requests per hour to avoid throttling.
"""

# Analysis:
# ✅ Correct: endpoint name, parameters, auth requirement, JSON format, rate limit
# ⚠️  Inferred: "Authorization header" (implied but not stated)
# ⚠️  Inferred: "optimized for performance" (not mentioned)
# ⚠️  Inferred: "avoid throttling" (implied but not stated)

context = {
    "source_text": source_text,
    "output_text": output_text
}

print("📝 Test case prepared - An ambiguous output with mixed facts and inferences")

📝 Test case prepared - An ambiguous output with mixed facts and inferences


## 🎬 Run the Evaluation

Let's see how our diverse jury handles this ambiguous case!

In [7]:
# Create the Groundedness metric
metric = GroundednessMetric()

print("🎬 Running evaluation with 5 judges...\n")
print("This may take 10-20 seconds as we query multiple APIs in parallel...\n")

# Evaluate
result = jury.evaluate(
    context=context,
    output=output_text,
    metric=metric
)

print("✅ Evaluation complete!")

🎬 Running evaluation with 5 judges...

This may take 10-20 seconds as we query multiple APIs in parallel...

✅ Evaluation complete!


## 📊 The Verdict: Consensus Results

Let's see what the jury decided collectively.

In [8]:
print("="*70)
print("⚖️  JURY VERDICT")
print("="*70)
print(f"\n🎯 Final Score: {result.final_score}/5")
print(f"✅ Valid: {result.is_valid}")
print(f"🎲 Confidence: {result.confidence:.2%}")
print(f"\n💡 Recommendation:\n   {result.get_recommendation()}")
print("\n" + "="*70)

# Get aggregation details
metadata = result.manifest.metadata
print(f"\n📈 Aggregation Details:")
print(f"   Strategy: {metadata.get('strategy_used', 'N/A')}")
print(f"   Vote Distribution: {metadata.get('aggregation_metadata', {}).get('vote_distribution', 'N/A')}")
print(f"   Total Votes: {metadata.get('aggregation_metadata', {}).get('total_votes', 'N/A')}")

⚖️  JURY VERDICT

🎯 Final Score: 4.0/5
✅ Valid: True
🎲 Confidence: 40.00%

💡 Recommendation:


📈 Aggregation Details:
   Strategy: MajorityVoting
   Vote Distribution: {4: 2, 5: 1, 3: 2}
   Total Votes: 5


## 🔍 Individual Judge Analysis

Now for the interesting part - let's see **exactly** how each judge reasoned about this case!

In [9]:
print("\n" + "="*70)
print("🧑‍⚖️  INDIVIDUAL JUDGE SCORES & REASONING")
print("="*70 + "\n")

for i, score in enumerate(result.manifest.individual_scores, 1):
    print(f"{'─'*70}")
    print(f"Judge {i}: {score.judge_id}")
    print(f"{'─'*70}")
    print(f"Score: {score.score}/5\n")
    print(f"Reasoning:")
    print(f"{score.reasoning}\n")


🧑‍⚖️  INDIVIDUAL JUDGE SCORES & REASONING

──────────────────────────────────────────────────────────────────────
Judge 1: Llama-3.1-8B
──────────────────────────────────────────────────────────────────────
Score: 4.0/5

Reasoning:
The model output is mostly supported by the source text, with minor rephrasing that keeps the meaning. The output accurately conveys the information about query parameters, authentication, and rate limiting. However, it introduces a new detail about the endpoint being "optimized for performance," which is not explicitly mentioned in the source text. Additionally, the output uses more user-friendly language, such as "find users by name or email address," whereas the source text uses more technical terms like "accepts query parameters 'name' and 'email'."

──────────────────────────────────────────────────────────────────────
Judge 2: Llama-3.3-70B
──────────────────────────────────────────────────────────────────────
Score: 4.0/5

Reasoning:
The output is mo

## 📊 Visual Comparison: Score Distribution

Let's visualize how the judges voted to see the diversity (or consensus) in their assessments.

In [10]:
# Create a DataFrame for easy comparison
judge_data = []
for score in result.manifest.individual_scores:
    judge_data.append({
        'Judge': score.judge_id,
        'Score': score.score,
        'Key Concern': score.reasoning.split('.')[0][:80] + '...' if len(score.reasoning.split('.')[0]) > 80 else score.reasoning.split('.')[0]
    })

df = pd.DataFrame(judge_data)

print("\n📊 JUDGE COMPARISON TABLE")
print("="*70)
display(df)

# Score statistics
print(f"\n📈 Score Statistics:")
print(f"   Mean: {df['Score'].mean():.2f}")
print(f"   Median: {df['Score'].median():.2f}")
print(f"   Std Dev: {df['Score'].std():.2f}")
print(f"   Range: {df['Score'].min():.0f} - {df['Score'].max():.0f}")


📊 JUDGE COMPARISON TABLE


,Judge,Score,Key Concern
0,Llama-3.1-8B,4.0,The model output is mostly supported by the so...
1,Llama-3.3-70B,4.0,The output is mostly supported by the source t...
2,GPT-3.5-Turbo,5.0,The model output is fully supported by the sou...
3,GPT-4o,3.0,The model output is mostly supported by the so...
4,Gemini-2.5-Flash,3.0,The model output is mostly supported by the so...



📈 Score Statistics:
   Mean: 3.80
   Median: 4.00
   Std Dev: 0.84
   Range: 3 - 5


## 🎯 Understanding Confidence Score

The **confidence score** tells us how much the judges agreed. Let's break it down:

In [11]:
# Extract vote distribution
vote_dist = metadata.get('aggregation_metadata', {}).get('vote_distribution', {})
total_votes = metadata.get('aggregation_metadata', {}).get('total_votes', 0)

print("\n🎲 CONFIDENCE CALCULATION")
print("="*70)
print(f"\nHow Majority Voting Confidence Works:")
print(f"   Confidence = (Votes for Winner) / (Total Judges)\n")

if vote_dist:
    print(f"Vote Breakdown:")
    for score_value, count in sorted(vote_dist.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total_votes * 100) if total_votes > 0 else 0
        bar = "█" * int(percentage / 5)
        print(f"   Score {score_value}: {count} judges ({percentage:.1f}%) {bar}")
    
    # Get the winner
    winner_score = result.final_score
    winner_votes = vote_dist.get(int(winner_score), 0)
    
    print(f"\n📊 Final Calculation:")
    print(f"   Winner Score: {winner_score}")
    print(f"   Votes for Winner: {winner_votes}")
    print(f"   Total Judges: {total_votes}")
    print(f"   Confidence: {winner_votes}/{total_votes} = {result.confidence:.2%}")
    
    # Interpretation
    print(f"\n💡 Interpretation:")
    if result.confidence >= 0.8:
        print(f"   🟢 STRONG CONSENSUS - Judges are highly aligned")
    elif result.confidence >= 0.6:
        print(f"   🟡 MODERATE CONSENSUS - Majority agree, but some dissent")
    elif result.confidence >= 0.4:
        print(f"   🟠 WEAK CONSENSUS - Significant disagreement among judges")
    else:
        print(f"   🔴 NO CONSENSUS - Judges are heavily divided")


🎲 CONFIDENCE CALCULATION

How Majority Voting Confidence Works:
   Confidence = (Votes for Winner) / (Total Judges)

Vote Breakdown:
   Score 4: 2 judges (40.0%) ████████
   Score 3: 2 judges (40.0%) ████████
   Score 5: 1 judges (20.0%) ████

📊 Final Calculation:
   Winner Score: 4.0
   Votes for Winner: 2
   Total Judges: 5
   Confidence: 2/5 = 40.00%

💡 Interpretation:
   🟠 WEAK CONSENSUS - Significant disagreement among judges


## 🔬 Why Did Judges Disagree?

Let's analyze the **reasoning diversity** to understand the different perspectives.

In [12]:
print("\n🔬 REASONING ANALYSIS")
print("="*70)

# Group judges by score
scores_grouped = {}
for score in result.manifest.individual_scores:
    score_val = score.score
    if score_val not in scores_grouped:
        scores_grouped[score_val] = []
    scores_grouped[score_val].append(score)

# Show reasoning patterns for each score group
for score_val in sorted(scores_grouped.keys(), reverse=True):
    judges_in_group = scores_grouped[score_val]
    print(f"\n📍 Judges who gave {score_val}/5:")
    for judge_score in judges_in_group:
        print(f"   • {judge_score.judge_id}")
    
    print(f"\n   Common themes in their reasoning:")
    # Extract first sentence of reasoning as key point
    for judge_score in judges_in_group:
        key_point = judge_score.reasoning.split('.')[0]
        print(f"   - {key_point}")
    print()


🔬 REASONING ANALYSIS

📍 Judges who gave 5.0/5:
   • GPT-3.5-Turbo

   Common themes in their reasoning:
   - The model output is fully supported by the source text


📍 Judges who gave 4.0/5:
   • Llama-3.1-8B
   • Llama-3.3-70B

   Common themes in their reasoning:
   - The model output is mostly supported by the source text, with minor rephrasing that keeps the meaning
   - The output is mostly supported by the source text, with minor rephrasing that keeps the meaning


📍 Judges who gave 3.0/5:
   • GPT-4o
   • Gemini-2.5-Flash

   Common themes in their reasoning:
   - The model output is mostly supported by the source text, but it includes a detail not found in the source: the claim that the endpoint is "optimized for performance
   - The model output is mostly supported by the source text, but it includes one piece of information that is not present in the source



## 🆚 Comparison: Single Judge vs Jury

Let's compare what would happen if we **only used one judge** versus the full jury.

In [13]:
print("\n🆚 SINGLE JUDGE VS JURY COMPARISON")
print("="*70)

# Test with each judge individually
single_judge_results = []

for judge in [judge_1, judge_2, judge_3, judge_4, judge_5]:
    single_jury = JuryEvaluator(judges=[judge], strategy=MajorityVoting())
    single_result = single_jury.evaluate(context, output_text, metric)
    
    single_judge_results.append({
        'Judge': judge.name,
        'Score': single_result.final_score,
        'Valid': single_result.is_valid,
        'Confidence': f"{single_result.confidence:.0%}"  # Always 100% for single judge
    })

# Add jury result
single_judge_results.append({
    'Judge': '🏆 JURY (5 Judges)',
    'Score': result.final_score,
    'Valid': result.is_valid,
    'Confidence': f"{result.confidence:.0%}"
})

comparison_df = pd.DataFrame(single_judge_results)
display(comparison_df)

print("\n💡 Key Insights:")
print("   • Single judges have 100% confidence (no disagreement to measure)")
print("   • Different judges gave different scores for the SAME input")
print("   • The jury provides a balanced view across all perspectives")
print("   • Confidence score reveals how much agreement exists")


🆚 SINGLE JUDGE VS JURY COMPARISON


,Judge,Score,Valid,Confidence
0,Llama-3.3-70B,4.0,True,100%
1,Llama-3.1-8B,4.0,True,100%
2,GPT-4o,3.0,True,100%
3,GPT-3.5-Turbo,5.0,True,100%
4,Gemini-2.5-Flash,2.0,False,100%
5,🏆 JURY (5 Judges),4.0,True,40%



💡 Key Insights:
   • Single judges have 100% confidence (no disagreement to measure)
   • Different judges gave different scores for the SAME input
   • The jury provides a balanced view across all perspectives
   • Confidence score reveals how much agreement exists


## 🧪 Experiment: Testing Edge Cases

Let's test two extreme cases to see how consensus behaves:
1. **Clear Success**: Output perfectly grounded
2. **Clear Failure**: Output with obvious hallucinations

In [14]:
# Test Case 1: Perfect Groundedness
perfect_output = """
The /v2/users/search endpoint accepts 'name' and 'email' query parameters. 
It returns a JSON array of user objects. Authentication is required via Bearer token. 
Rate limiting is 100 requests per hour per API key.
"""

perfect_context = {
    "source_text": source_text,
    "output_text": perfect_output
}

print("🧪 TEST 1: Perfect Groundedness")
print("="*70)
perfect_result = jury.evaluate(perfect_context, perfect_output, metric)

print(f"\n📊 Results:")
print(f"   Final Score: {perfect_result.final_score}/5")
print(f"   Confidence: {perfect_result.confidence:.0%}")
print(f"   Valid: {perfect_result.is_valid}")

# Count how many judges gave top scores
perfect_scores = [s.score for s in perfect_result.manifest.individual_scores]
high_scores = sum(1 for s in perfect_scores if s >= 4)
print(f"   Judges giving 4-5 stars: {high_scores}/{len(perfect_scores)}")

🧪 TEST 1: Perfect Groundedness

📊 Results:
   Final Score: 4.0/5
   Confidence: 60%
   Valid: True
   Judges giving 4-5 stars: 5/5


In [15]:
# Test Case 2: Clear Hallucination
hallucinated_output = """
The /v2/users/search endpoint supports full-text search with regex patterns. 
It can return up to 1000 results per query and includes pagination with cursor-based navigation. 
The endpoint is free to use with unlimited requests and supports WebSocket connections for real-time updates.
"""

hallucinated_context = {
    "source_text": source_text,
    "output_text": hallucinated_output
}

print("\n🧪 TEST 2: Clear Hallucination")
print("="*70)
hallucinated_result = jury.evaluate(hallucinated_context, hallucinated_output, metric)

print(f"\n📊 Results:")
print(f"   Final Score: {hallucinated_result.final_score}/5")
print(f"   Confidence: {hallucinated_result.confidence:.0%}")
print(f"   Valid: {hallucinated_result.is_valid}")

# Count how many judges gave low scores
hallucinated_scores = [s.score for s in hallucinated_result.manifest.individual_scores]
low_scores = sum(1 for s in hallucinated_scores if s <= 2)
print(f"   Judges giving 1-2 stars: {low_scores}/{len(hallucinated_scores)}")


🧪 TEST 2: Clear Hallucination

📊 Results:
   Final Score: 1.0/5
   Confidence: 60%
   Valid: False
   Judges giving 1-2 stars: 5/5


## 📈 Summary: Edge Cases vs Ambiguous Cases

In [16]:
summary_data = [
    {
        'Case': '✅ Perfect Groundedness',
        'Score': f"{perfect_result.final_score}/5",
        'Confidence': f"{perfect_result.confidence:.0%}",
        'Agreement': 'HIGH' if perfect_result.confidence >= 0.8 else 'MODERATE'
    },
    {
        'Case': '⚠️  Ambiguous (Original)',
        'Score': f"{result.final_score}/5",
        'Confidence': f"{result.confidence:.0%}",
        'Agreement': 'HIGH' if result.confidence >= 0.8 else 'MODERATE' if result.confidence >= 0.6 else 'LOW'
    },
    {
        'Case': '❌ Clear Hallucination',
        'Score': f"{hallucinated_result.final_score}/5",
        'Confidence': f"{hallucinated_result.confidence:.0%}",
        'Agreement': 'HIGH' if hallucinated_result.confidence >= 0.8 else 'MODERATE'
    }
]

summary_df = pd.DataFrame(summary_data)

print("\n📊 CONSENSUS BEHAVIOR ACROSS DIFFERENT CASES")
print("="*70)
display(summary_df)

print("\n💡 Key Observations:")
print("   • Clear cases (perfect/hallucinated) → HIGH confidence")
print("   • Ambiguous cases → MODERATE/LOW confidence")
print("   • Low confidence is VALUABLE - it signals uncertainty!")
print("   • You can use confidence to trigger human review on edge cases")


📊 CONSENSUS BEHAVIOR ACROSS DIFFERENT CASES


,Case,Score,Confidence,Agreement
0,✅ Perfect Groundedness,4.0/5,60%,MODERATE
1,⚠️ Ambiguous (Original),4.0/5,40%,LOW
2,❌ Clear Hallucination,1.0/5,60%,MODERATE



💡 Key Observations:
   • Clear cases (perfect/hallucinated) → HIGH confidence
   • Ambiguous cases → MODERATE/LOW confidence
   • Low confidence is VALUABLE - it signals uncertainty!
   • You can use confidence to trigger human review on edge cases


## 🎯 Key Takeaways

### Why Multiple Judges Matter:

1. **Reduced Bias** 🎭
   - Different models have different training data and biases
   - Consensus averages out individual model quirks
   - More robust across diverse evaluation scenarios

2. **Confidence Metrics** 📊
   - Single judges are always "100% confident" (even when wrong)
   - Jury confidence reveals actual agreement level
   - Low confidence = valuable signal to trigger human review

3. **Transparency** 🔍
   - See individual reasoning from each judge
   - Understand WHY judges agreed or disagreed
   - Debug evaluation behavior more easily

4. **Reliability** ✅
   - Extreme cases (perfect/terrible) → High consensus
   - Ambiguous cases → Lower consensus (as expected!)
   - This matches human behavior patterns

---

## 💡 Pro Tips for Production

1. **Use 3-5 judges** for good balance (more = diminishing returns)
2. **Mix model sizes** (large + small) for efficiency
3. **Set confidence thresholds** for auto-routing to human review
4. **Log all reasoning** for debugging and improvement
5. **Consider cost** when choosing judges (Groq is very affordable!)

---

**Happy Evaluating! 🎯**